<h1>Data Analysis</h1>

In [1]:
#!pip install kagglehub

In [2]:
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from datetime import datetime
from sklearn import linear_model
from sklearn.model_selection import train_test_split
from datetime import datetime
from sklearn.metrics import r2_score,mean_squared_error, root_mean_squared_error

In [3]:
#import kagglehub

# Download latest version
#path = kagglehub.dataset_download("ahmedshahriarsakib/usa-real-estate-dataset")

#print("Path to dataset files:", path)

In [4]:
#Download data into dataframe
data_df = pd.read_csv('Datasets/realtor-data.zip.csv',dtype={'zip_code' : str})
data_df.head(8)

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,NaN
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,NaN
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,NaN
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,NaN
4,34632.0,for_sale,65000.0,6.0,2.0,0.05,331151.0,Mayaguez,Puerto Rico,00680,NaN,NaN
5,103378.0,for_sale,179000.0,4.0,3.0,0.46,1850806.0,San Sebastian,Puerto Rico,00612,2520.0,NaN
6,1205.0,for_sale,50000.0,3.0,1.0,0.20,1298094.0,Ciales,Puerto Rico,00639,2040.0,NaN
7,50739.0,for_sale,71600.0,3.0,2.0,0.08,1048466.0,Ponce,Puerto Rico,00731,1050.0,NaN


In [5]:
crime_risk_stats_df = pd.read_csv(
    "datasets/(Copy) VT/usa_pl_2024 crimerisk-statistics.csv"
)

crime_risk_stats_df.head()

,VARIABLE,DESCRIPTION,SUM,AVERAGE,MAXIMUM,MINIMUM,ZERO_RECS
0,CRMA4PERC,Personal Crime,31021389,968.298811,908062,0,442
1,CRMA4MURD,Murder,30843375,962.742298,892768,0,464
2,CRMA4RAPE,Rape,31012348,968.016606,900269,0,461
3,CRMA4ROBB,Robbery,31028107,968.508506,921872,0,492
4,CRMA4ASST,Assault,31066074,969.693604,922302,0,461


In [6]:
sq_ft_max = data_df.house_size.max()
sq_ft_min = data_df.house_size.min()

<h4>Cleaning data</h4>

In [7]:
#find null data in house size, zip code, state, city
data_df_clean = data_df.dropna(subset=['brokered_by', 'status', 'price','bed','bath','acre_lot','street','city','state','zip_code','house_size'])
data_df_clean.shape

(1354105, 12)

In [8]:
#data_df_clean['zip_code']= data_df_clean['zip_code'].astype(str)
temp_df = pd.read_csv("datasets/(Copy) VT/usa_zi_premium_environment.csv",dtype={"ID":str})
#firstelem = data_df_clean.loc[0]
#firstelem['zip_code']

In [9]:
#Encoding
from sklearn.preprocessing import OneHotEncoder

In [10]:
enc = OneHotEncoder(handle_unknown='ignore')
#Encode status
status = {'status' : data_df_clean.status}
statusdf = pd.DataFrame(status)
status_ohe = pd.get_dummies(statusdf, dtype = int)
#print(status_ohe)
data_df_clean['prev_sold_date'].fillna('1970-1-2', inplace=True)

C:\Users\omars\AppData\Local\Temp\ipykernel_40708\1339336517.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_df_clean['prev_sold_date'].fillna('1970-1-2', inplace=True)


In [11]:
stateList = []
for i in data_df_clean['state']:
    if i not in stateList:
        stateList.append(i)

def state_binary(state):
    stateBin = []
    for item in stateList:
        if item == state:
            stateBin.append(1)
        else:
            stateBin.append(0)
    return stateBin

In [12]:
#state
#onehot encode state

data_df_clean['state_bin'] = data_df_clean['state'].apply(lambda x: state_binary(x))
#city


C:\Users\omars\AppData\Local\Temp\ipykernel_40708\2665686296.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_df_clean['state_bin'] = data_df_clean['state'].apply(lambda x: state_binary(x))


In [13]:
#current = datetime.now()
#current.timestamp()
data_df_clean.head()

,brokered_by,status,price,bed,bath,acre_lot,street,city,state,zip_code,house_size,prev_sold_date,state_bin
0,103378.0,for_sale,105000.0,3.0,2.0,0.12,1962661.0,Adjuntas,Puerto Rico,00601,920.0,1970-1-2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,52707.0,for_sale,80000.0,4.0,2.0,0.08,1902874.0,Adjuntas,Puerto Rico,00601,1527.0,1970-1-2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,103379.0,for_sale,67000.0,2.0,1.0,0.15,1404990.0,Juana Diaz,Puerto Rico,00795,748.0,1970-1-2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,31239.0,for_sale,145000.0,4.0,2.0,0.10,1947675.0,Ponce,Puerto Rico,00731,1800.0,1970-1-2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
5,103378.0,for_sale,179000.0,4.0,3.0,0.46,1850806.0,San Sebastian,Puerto Rico,00612,2520.0,1970-1-2,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [14]:
#concatenate and drop
data_df_clean = data_df_clean.drop(labels=['status','state', 'city'], axis = 'columns')
data_df_clean = pd.concat([data_df_clean,status_ohe, status_ohe], axis = 'columns')
data_df_clean.drop(labels=['prev_sold_date'],inplace=True, axis=1)
def convert_date(date_str):
    newdt = datetime.strptime(date_str, "%Y-%m-%d")
    
    print(newdt)
    return newdt.timestamp()
#Convert data to pandas datatime
#data_df_clean['prev_sold_date'] =data_df_clean['prev_sold_date'].apply(convert_date)


In [15]:
#split into X and Y
X = data_df_clean.drop(labels=['price'], axis = 'columns')
Y = data_df_clean['price']
X_train, X_test, y_train, y_test = train_test_split(X,Y, test_size=0.33, random_state=42, shuffle=True)


In [16]:
clf = linear_model.Lasso(alpha=0.3)
#NOTE TO SELF: FIX THIS!
clf.fit(X_train,y_train)

ValueError: setting an array element with a sequence.

In [ ]:
y_pred = clf.predict(X_test)
y_test = y_test.to_numpy()
print("RMSE: ", root_mean_squared_error(y_test, y_pred))

TypeError: can't multiply sequence by non-int of type 'float'